# PharmaIQ Lite+ — Exploratory Data Analysis

**Synthetic demonstration data (`synthetic_v1`).** Production logic lives in `src/analytics/eda.py`; this notebook only calls it and displays results. Run `python scripts/bootstrap.py --skip-db` first.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
from src import config
from src.analytics import eda
panel = pd.read_parquet(config.PANEL_PARQUET)
panel.info()

## Structural checks: keys, row count, continuity, missingness

In [ ]:
keys = ['month_start', 'region_id', 'product_id']
print('rows', len(panel), '| duplicate keys', panel.duplicated(keys).sum())
print('months', panel['month_start'].nunique(), '| per series', panel.groupby(['region_id','product_id']).size().unique())
print('missing values per column:'); print(panel.isna().sum()[panel.isna().sum() > 0] if panel.isna().any().any() else 'none')
print('zero-unit rows', (panel['units_sold'] == 0).sum())
panel.describe().T

## Figures and summary statistics

In [ ]:
stats = eda.run_eda(panel)
print(json.dumps({k: v for k, v in stats.items() if k != 'figures'}, indent=2, default=str))
for f in stats['figures']:
    display(Image(filename=str(config.FIGURES_DIR / f)))

## Interpretation notes
Documented findings are in `docs/business_case.md` and are taken from `artifacts/metrics/eda_summary.json`. Spend vs units is descriptive only: in this generator campaign months raise both spend and units, so the correlation is confounded.